# RAG from scratch with LangChain, Ollama and Groq

My notes and code while working through the *RAG from Scratch* video series by Lance Martin (LangChain). The videos use OpenAI and LangSmith. I wanted to do the same thing without either, so this version uses:

- Ollama (`nomic-embed-text`) for embeddings, running on my PC
- Chroma as the vector store, in memory
- Groq for the LLM that writes the final answer

Rough flow:

```
documents -> split into chunks -> embed -> vector store        (indexing, once)
question  -> embed -> find closest chunks                      (retrieval)
question + chunks -> prompt -> LLM -> answer                   (generation)
```

Covers the first four videos: overview, indexing, retrieval, generation.

## Setup

Need the Ollama app running and a Groq API key (console.groq.com).

In [ ]:
%pip install -qU langchain langchain-core langchain-community langchain-groq langchain-ollama langchain-chroma langchain-text-splitters ollama chromadb tiktoken beautifulsoup4 numpy

In [ ]:
# pull the embedding model (needs the Ollama app open)
from ollama import pull
pull("nomic-embed-text")

In [ ]:
# key is typed in at runtime so it never gets saved in the notebook
import getpass, os

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API key: ")

os.environ["USER_AGENT"] = "rag-from-scratch-notebook"

Two objects used throughout: `embd` for embeddings and `llm` for answers. If Groq says the model isn't available, change `GROQ_MODEL` (current list is on console.groq.com/docs/models).

I also put a few helpers here so the later cells stay short.

In [ ]:
from langchain_ollama import OllamaEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_chroma import Chroma

GROQ_MODEL = "openai/gpt-oss-20b"

embd = OllamaEmbeddings(model="nomic-embed-text")
llm = ChatGroq(model=GROQ_MODEL, temperature=0)

# same text as the rlm/rag-prompt on LangChain Hub, kept local (the Hub needs LangSmith)
def get_rag_prompt():
    return ChatPromptTemplate.from_template(
        "You are an assistant for question-answering tasks. Use the following pieces "
        "of retrieved context to answer the question. If you don't know the answer, "
        "just say that you don't know. Use three sentences maximum and keep the "
        "answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"
    )

# Chroma is in memory, so re-running a cell would add the chunks twice.
# Delete the old collection first.
def build_vectorstore(splits, name):
    try:
        Chroma(collection_name=name, embedding_function=embd).delete_collection()
    except Exception:
        pass
    return Chroma.from_documents(documents=splits, embedding=embd, collection_name=name)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

## Part 1: Overview

Main points from the video:

- Most data is private, so the model has never seen it.
- Context windows are limited, so we can't paste everything in. Only the relevant parts should go in.
- RAG = index the documents, retrieve what's relevant to the question, then let the LLM answer using that.

Here is the whole pipeline in one cell, following the quickstart. Parts 2 to 4 break it down.

In [ ]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# indexing
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(class_=("post-content", "post-title", "post-header"))
    ),
)
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

vectorstore = build_vectorstore(splits, "part1_quickstart")
retriever = vectorstore.as_retriever()

# retrieval + generation
prompt = get_rag_prompt()

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(rag_chain.invoke("What is Task Decomposition?"))

## Part 2: Indexing

Steps: load, split, embed, store.

### Why embeddings instead of word counts

The video compares two ways of turning text into numbers. Bag of words gives a sparse vector (mostly zeros) and matches on exact words. An embedding gives a dense vector and matches on meaning.

To check this myself I used a question and a document that mean related things but share no words.

In [ ]:
question = "What kinds of pets do I like?"
document = "My favorite pet is a cat."
unrelated = "The stock market fell sharply on Monday."

In [ ]:
import re
import numpy as np
from collections import Counter

def tokenize(text):
    return re.findall(r"[a-z']+", text.lower())

texts = {"question": question, "document": document, "unrelated": unrelated}
vocab = sorted({w for t in texts.values() for w in tokenize(t)})

def bag_of_words(text):
    counts = Counter(tokenize(text))
    return [counts[w] for w in vocab]

bow = {name: bag_of_words(t) for name, t in texts.items()}
print("vocab size:", len(vocab))
print("question:", bow["question"])
print("document:", bow["document"])

In [ ]:
def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product / (norm_vec1 * norm_vec2)

# 1 = same direction, 0 = nothing in common
print("bag of words, question vs document:",
      round(cosine_similarity(bow["question"], bow["document"]), 3))

Zero, because "pets" and "pet" are different words to a word counter. Now the same pair with embeddings.

### Embeddings

In [ ]:
query_result = embd.embed_query(question)
document_result = embd.embed_query(document)
unrelated_result = embd.embed_query(unrelated)

print("embedding length:", len(query_result))
print("first 5 values:", [round(x, 4) for x in query_result[:5]])

In [ ]:
print("question vs document :", round(cosine_similarity(query_result, document_result), 3))
print("question vs unrelated:", round(cosine_similarity(query_result, unrelated_result), 3))

The related pair scores clearly higher than the unrelated one, even with no shared words. That is why the vector store uses embeddings.

### Counting tokens

Models read tokens, not characters (roughly 4 characters per token). This matters for context window limits and cost. `tiktoken` is OpenAI's tokenizer, so the count is only an estimate for other models.

In [ ]:
import tiktoken

def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    return len(encoding.encode(string))

num_tokens_from_string(question, "cl100k_base")

### Loading documents

A loader pulls text from a source. Here it's a web page. LangChain has loaders for PDFs, CSVs, YouTube, Notion and many others (integrations.langchain.com).

In [ ]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(class_=("post-content", "post-title", "post-header"))
    ),
)
blog_docs = loader.load()

print("documents:", len(blog_docs))
print("characters:", len(blog_docs[0].page_content))
print("metadata:", blog_docs[0].metadata)

### Splitting

One long document is too big to search well, so it gets split into chunks. `RecursiveCharacterTextSplitter` tries to break on paragraphs first, then lines, then words. I used 300 tokens per chunk with 50 tokens of overlap so a sentence isn't lost at the edge of a chunk.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,
    chunk_overlap=50,
)
splits = text_splitter.split_documents(blog_docs)

print("chunks:", len(splits))
print()
print(splits[0].page_content)

### Vector store

Chroma stores each chunk with its embedding. The retriever is the search interface on top of it.

In [ ]:
vectorstore = build_vectorstore(splits, "part2_indexing")
retriever = vectorstore.as_retriever()
print("indexed", len(splits), "chunks")

## Part 3: Retrieval

The question gets embedded with the same model, then the vector store returns the chunks whose vectors are closest. `k` is how many chunks to return. Starting with `k=1`.

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

In [ ]:
# .invoke() replaces the older get_relevant_documents()
docs = retriever.invoke("What is Task Decomposition?")

In [ ]:
len(docs)

In [ ]:
for d in docs:
    print(d.page_content)
    print("\nmetadata:", d.metadata)

To see how close each chunk is, `similarity_search_with_score` returns a score too. In Chroma this is a distance, so a lower number means a closer match.

In [ ]:
results = vectorstore.similarity_search_with_score("What is Task Decomposition?", k=3)

for i, (doc, score) in enumerate(results, start=1):
    print(f"#{i}  distance = {score:.3f}")
    print(doc.page_content[:200].replace("\n", " "), "...\n")

Changing `k` to see the difference:

In [ ]:
for k in [1, 3, 5]:
    r = vectorstore.as_retriever(search_kwargs={"k": k})
    print(f"k={k}: {len(r.invoke('What is Task Decomposition?'))} chunks")

## Part 4: Generation

The retrieved chunks and the question go into a prompt, the prompt goes to the LLM, and a parser turns the reply into a plain string.

```
{context, question} -> prompt -> PromptValue -> LLM -> ChatMessage -> parser -> string
```

### Prompt

The wording "based only on the following context" is what keeps the model from answering from its own memory.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)
prompt

In [ ]:
# quick check that Groq is working
llm.invoke("Say hello in five words.").content

### Chain

`|` passes the output of the left side into the right side, so `prompt | llm` fills in the prompt and sends it to the model.

In [ ]:
chain = prompt | llm

In [ ]:
result = chain.invoke({"context": docs, "question": "What is Task Decomposition?"})
print(result.content)

Running the same thing one stage at a time to see what each piece returns:

In [ ]:
from langchain_core.output_parsers import StrOutputParser

inputs = {"context": format_docs(docs), "question": "What is Task Decomposition?"}

prompt_value = prompt.invoke(inputs)
print("1.", type(prompt_value).__name__)
print(prompt_value.to_string()[:300], "...\n")

chat_message = llm.invoke(prompt_value)
print("2.", type(chat_message).__name__)

answer = StrOutputParser().invoke(chat_message)
print("3.", type(answer).__name__)
print()
print(answer)

### The community RAG prompt

The video uses a ready-made prompt from LangChain Hub (`rlm/rag-prompt`). I'm using a local copy of it, so this one is for comparing with my own prompt above.

In [ ]:
prompt_hub_rag = get_rag_prompt()
prompt_hub_rag

### Full RAG chain

Retriever finds the chunks, `format_docs` joins them into one string, `RunnablePassthrough()` passes the question through unchanged, then prompt, LLM and parser. Using `k=3` here so the model gets a bit more context.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(rag_chain.invoke("What is Task Decomposition?"))

### Other questions

The second question isn't covered by the blog, so with this prompt the model should say it doesn't know.

In [ ]:
for q in [
    "What are the main components of an LLM-powered agent?",
    "What is the capital of France?",
]:
    print("Q:", q)
    print("A:", rag_chain.invoke(q))
    print()

## Summary

- Indexing: `WebBaseLoader` -> `RecursiveCharacterTextSplitter` -> `OllamaEmbeddings` -> `Chroma`
- Retrieval: embed the question, return the `k` closest chunks
- Generation: `ChatPromptTemplate` | `ChatGroq` | `StrOutputParser`

Common errors I ran into or expect to:

- Ollama connection error: the Ollama app isn't open
- model "nomic-embed-text" not found: run the pull cell again
- Groq 401: wrong or missing API key
- Groq model not found: change `GROQ_MODEL`

Next in the series: query translation (multi-query, RAG-Fusion, decomposition, step-back, HyDE), routing, query construction, and more advanced indexing and retrieval.

Based on the RAG from Scratch series by Lance Martin, LangChain.